# Korean Chatbot - Stage 1 학습 노트북
KoAlpaca 데이터셋으로 스크래치 Transformer 학습

## 0. 환경 설정

In [ ]:
!pip install datasets tqdm -q

In [ ]:
import os

REPO_URL = "https://github.com/kkkk2058/korean-chatbot"  # 본인 레포로 수정
REPO_DIR = "korean-chatbot"
STAGE_DIR = f"{REPO_DIR}/stage1_from_scratch"

if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL
else:
    !git -C $REPO_DIR pull

import sys
sys.path.insert(0, STAGE_DIR)
print("레포 준비 완료")

## 1. 토크나이저 준비
저장된 tokenizer.json 이 있으면 로드, 없으면 KoAlpaca로 새로 학습

In [ ]:
import sys, importlib, shutil
sys.path.insert(0, STAGE_DIR)

import src.tokenizer
importlib.reload(src.tokenizer)

from src.tokenizer import BPETokenizer, train_on_koalpaca
import config

TOKENIZER_SAVE_PATH = "tokenizer.json"
DRIVE_DIR = "/content/drive/MyDrive/korean_chatbot"

if not os.path.exists(TOKENIZER_SAVE_PATH) and os.path.exists(f"{DRIVE_DIR}/tokenizer.json"):
    shutil.copy(f"{DRIVE_DIR}/tokenizer.json", TOKENIZER_SAVE_PATH)
    print("Drive에서 tokenizer.json 복사 완료")

tokenizer = BPETokenizer()

if os.path.exists(TOKENIZER_SAVE_PATH):
    tokenizer.load(TOKENIZER_SAVE_PATH)
    print(f"토크나이저 로드 완료. vocab size = {len(tokenizer.vocab)}")
else:
    tokenizer = train_on_koalpaca(
        vocab_size=config.VOCAB_SIZE,
        save_path=TOKENIZER_SAVE_PATH
    )
    print(f"토크나이저 학습·저장 완료. vocab size = {len(tokenizer.vocab)}")

## 2. 데이터셋 로드 및 전처리

In [ ]:
from datasets import load_dataset

print("KoAlpaca 데이터셋 로드 중...")
ds = load_dataset("beomi/KoAlpaca-v1.1a", split="train")
print(f"총 샘플 수: {len(ds)}")
print("예시:", ds[0])

In [ ]:
import torch, time
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm

PAD_ID = tokenizer.vocab["<pad>"]
BOS_ID = tokenizer.vocab["<s>"]
EOS_ID = tokenizer.vocab["</s>"]


class KoAlpacaDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, max_seq_len):
        self.samples = []
        for row in tqdm(hf_dataset, desc="데이터셋 변환 중"):
            inp = row.get("input", "").strip()
            text = f"{row['instruction'].strip()}\n{inp + chr(10) if inp else ''}{row['output'].strip()}"
            ids = [BOS_ID] + tokenizer.encode(text) + [EOS_ID]
            if len(ids) > 1:
                self.samples.append(ids[: max_seq_len + 1])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        ids = self.samples[idx]
        return torch.tensor(ids[:-1], dtype=torch.long), torch.tensor(ids[1:], dtype=torch.long)


def collate_fn(batch):
    xs, ys = zip(*batch)
    max_len = max(x.size(0) for x in xs)
    pad = lambda t: torch.nn.functional.pad(t, (0, max_len - t.size(0)), value=PAD_ID)
    return torch.stack([pad(x) for x in xs]), torch.stack([pad(y) for y in ys])


t0 = time.time()
dataset = KoAlpacaDataset(ds, tokenizer, max_seq_len=config.MAX_SEQ_LEN)
dataloader = DataLoader(dataset, batch_size=config.BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2, pin_memory=True)
print(f"샘플 {len(dataset)}개 / 배치 {len(dataloader)}개  ({time.time()-t0:.1f}초)")

## 3. 모델 초기화

In [ ]:
from src.model import Transformer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = Transformer(
    vocab_size=len(tokenizer.vocab),
    d_model=config.D_MODEL,
    n_heads=config.N_HEADS,
    n_layers=config.N_LAYERS,
    max_seq_len=config.MAX_SEQ_LEN,
    dropout=config.DROPOUT,
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"파라미터 수: {total_params:,} ({total_params/1e6:.1f}M)")

## 4. 학습

In [ ]:
import math, time
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

criterion = torch.nn.CrossEntropyLoss(ignore_index=PAD_ID)
optimizer = torch.optim.AdamW(model.parameters(), lr=config.LR)

total_steps = len(dataloader) * config.EPOCHS
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)

CHECKPOINT_PATH = "checkpoint.pt"
GRAD_CLIP = 1.0

# Drive에 체크포인트가 있으면 복사해서 이어 학습
if not os.path.exists(CHECKPOINT_PATH) and os.path.exists(f"{DRIVE_DIR}/checkpoint.pt"):
    shutil.copy(f"{DRIVE_DIR}/checkpoint.pt", CHECKPOINT_PATH)
    print("Drive에서 checkpoint.pt 복사 완료")

start_epoch = 0
if os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    start_epoch = ckpt["epoch"] + 1
    print(f"체크포인트 로드 완료. epoch {start_epoch}부터 재개")

history = {"epoch": [], "loss": [], "ppl": [], "elapsed": []}
train_start = time.time()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

def update_plot():
    for ax in axes:
        ax.cla()
    axes[0].plot(history["epoch"], history["loss"], "b-o", markersize=4)
    axes[0].set_title("Train Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].grid(True)

    axes[1].plot(history["epoch"], history["ppl"], "r-o", markersize=4)
    axes[1].set_title("Perplexity")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("PPL")
    axes[1].grid(True)

    elapsed = time.time() - train_start
    done_epochs = len(history["epoch"])
    remaining_epochs = config.EPOCHS - start_epoch - done_epochs
    eta_str = ""
    if done_epochs > 0:
        per_epoch = elapsed / done_epochs
        eta_sec = per_epoch * remaining_epochs
        eta_str = f"  |  ETA {eta_sec/60:.1f}분"
    fig.suptitle(f"경과 {elapsed/60:.1f}분{eta_str}", fontsize=12)

    fig.tight_layout()
    clear_output(wait=True)
    display(fig)


def train_epoch(epoch):
    model.train()
    total_loss = 0
    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{config.EPOCHS}", leave=False)
    for step, (x, y) in enumerate(pbar):
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        avg = total_loss / (step + 1)
        pbar.set_postfix({"loss": f"{avg:.4f}", "ppl": f"{math.exp(avg):.1f}"})
    return total_loss / len(dataloader)


print("학습 시작!")
for epoch in range(start_epoch, config.EPOCHS):
    avg_loss = train_epoch(epoch)
    ppl = math.exp(avg_loss)
    elapsed = time.time() - train_start
    history["epoch"].append(epoch + 1)
    history["loss"].append(avg_loss)
    history["ppl"].append(ppl)
    history["elapsed"].append(elapsed)
    update_plot()
    print(f"Epoch {epoch+1:02d} | loss {avg_loss:.4f} | ppl {ppl:.2f} | 경과 {elapsed/60:.1f}분")
    torch.save({"epoch": epoch, "model": model.state_dict(), "optimizer": optimizer.state_dict()}, CHECKPOINT_PATH)

print("학습 완료!")

## 5. 간단한 생성 테스트

In [ ]:
@torch.no_grad()
def generate(prompt, max_new_tokens=100, temperature=1.0, top_k=50):
    model.eval()
    ids = [BOS_ID] + tokenizer.encode(prompt)
    x = torch.tensor([ids], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        if x.size(1) >= config.MAX_SEQ_LEN:
            break
        logits = model(x)[:, -1, :]   # 마지막 토큰의 logit
        logits = logits / temperature

        # top-k 샘플링
        if top_k > 0:
            topk_vals, _ = torch.topk(logits, top_k)
            logits[logits < topk_vals[:, -1:]] = float("-inf")

        probs = torch.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        x = torch.cat([x, next_id], dim=1)

        if next_id.item() == EOS_ID:
            break

    generated_ids = x[0].tolist()[len(ids):]  # 프롬프트 제외
    return tokenizer.decode(generated_ids)


prompt = "한국의 수도는 어디인가요?"
print(f"입력: {prompt}")
print(f"생성: {generate(prompt)}")

## 6. Google Drive 저장 (선택)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
DRIVE_DIR = "/content/drive/MyDrive/korean_chatbot"
os.makedirs(DRIVE_DIR, exist_ok=True)

shutil.copy(CHECKPOINT_PATH, f"{DRIVE_DIR}/checkpoint.pt")
shutil.copy(TOKENIZER_SAVE_PATH, f"{DRIVE_DIR}/tokenizer.json")
print(f"Drive 저장 완료 → {DRIVE_DIR}")